# Simple multinomial-logit trading strategy example

This notebook is a minimal, self-contained version of the trading-strategy procedure. It uses simulated data so that every object can be printed and inspected.

We use one predictor, estimate a multinomial logit, predict out of sample, compute the majority-class benchmark, and fill a strategy-return vector according to the predicted class.

Class convention:

- `0 = SELL`
- `1 = HOLD`
- `2 = BUY`


## 1. Setup and simulated data

We simulate one explanatory variable $x_t$, a stock return $r_t^{stock}$, and a daily risk-free return $r_t^f$.

The multinomial-logit model is estimated with one regressor:

$$
\Pr(Y_t=k\mid x_t)=\frac{\exp(\alpha_k + \beta_k x_t)}{1 + \sum_{j=1}^{K-1}\exp(\alpha_j + \beta_j x_t)}, \qquad k=1,\ldots,K-1.
$$

The remaining class is the baseline class:

$$
\Pr(Y_t=K\mid x_t)
=
\frac{1}{1 + \sum_{j=1}^{K-1}\exp(\alpha_j + \beta_j x_t)}.
$$

In `statsmodels.MNLogit`, one class is treated as the baseline.

In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

np.random.seed(42)

n = 300
train_frac = 0.70

# Simulate one predictor
x = np.random.normal(size=n)

# Simulate stock returns with weak predictability from x
r_stock = 0.001 + 0.012 * x + np.random.normal(scale=0.025, size=n)

# Constant daily risk-free return, about 2.5% annualized
r_f = np.repeat(0.025 / 252, n)

df = pd.DataFrame({
    "x": x,
    "r_stock": r_stock,
    "r_f": r_f,
})

print("First rows of simulated data:")
print(df.head(10).round(4))

First rows of simulated data:
        x  r_stock     r_f
0  0.4967  -0.0138  0.0001
1 -0.1383  -0.0147  0.0001
2  0.6477   0.0275  0.0001
3  1.5230   0.0345  0.0001
4 -0.2342  -0.0023  0.0001
5 -0.2341   0.0011  0.0001
6  1.5792   0.0519  0.0001
7  0.7674  -0.0046  0.0001
8 -0.4695   0.0090  0.0001
9  0.5426   0.0025  0.0001


## 2. Create classes

For teaching purposes, classes are defined directly from the realized stock return.

$$
Y_t =\begin{cases}
0 & \text{SELL, if } r_t^{stock} < 0,\\
1 & \text{HOLD, if } 0 \le r_t^{stock} \le q_{0.67},\\
2 & \text{BUY, if } r_t^{stock} > q_{0.67},
\end{cases}
$$

where $q_{0.67}$ is the 67th percentile of returns in the training sample. In a real forecasting exercise, the label should be based on future returns and predictors should be known before the trading decision.

In [2]:
split = int(train_frac * n)

train = df.iloc[:split].copy()
test = df.iloc[split:].copy()

hi = train["r_stock"].quantile(0.67)

def make_class(r):
    if r < 0:
        return 0      # SELL
    elif r <= hi:
        return 1      # HOLD
    else:
        return 2      # BUY

train["Y"] = train["r_stock"].apply(make_class).astype(int)
test["Y"] = test["r_stock"].apply(make_class).astype(int)

print("Training threshold hi:", round(hi, 4))
print("\nTraining class counts:")
print(train["Y"].value_counts().sort_index())
print("\nTest class counts:")
print(test["Y"].value_counts().sort_index())
print("\nFirst rows after labeling:")
print(train[["x", "r_stock", "r_f", "Y"]].head(10).round(4))

Training threshold hi: 0.0129

Training class counts:
Y
0    105
1     36
2     69
Name: count, dtype: int64

Test class counts:
Y
0    50
1    15
2    25
Name: count, dtype: int64

First rows after labeling:
        x  r_stock     r_f  Y
0  0.4967  -0.0138  0.0001  0
1 -0.1383  -0.0147  0.0001  0
2  0.6477   0.0275  0.0001  2
3  1.5230   0.0345  0.0001  2
4 -0.2342  -0.0023  0.0001  0
5 -0.2341   0.0011  0.0001  1
6  1.5792   0.0519  0.0001  2
7  0.7674  -0.0046  0.0001  0
8 -0.4695   0.0090  0.0001  1
9  0.5426   0.0025  0.0001  1


## 3. Estimate the multinomial logit

The model uses only one regressor:

$$
Y_t \sim x_t.
$$

The design matrix is:

$$
X_t = (1, x_t),
$$

where the first column is the intercept.

In [3]:
X_train = sm.add_constant(train[["x"]], has_constant="add")
y_train = train[["Y"]]

print("X_train:")
print(X_train.head().round(4))
print("\ny_train:")
print(y_train.head(10))

model = sm.MNLogit(y_train, X_train)
res = model.fit(method="newton", maxiter=100, disp=False)

print("\nEstimated coefficients:")
coefficients = pd.DataFrame(res.params.round(4))
coefficients.columns = ["Y=1", "Y=2"]
print(coefficients)

X_train:
   const       x
0    1.0  0.4967
1    1.0 -0.1383
2    1.0  0.6477
3    1.0  1.5230
4    1.0 -0.2342

y_train:
   Y
0  0
1  0
2  2
3  2
4  0
5  1
6  2
7  0
8  1
9  1

Estimated coefficients:
          Y=1     Y=2
const -0.9987 -0.4461
x      0.6823  1.0171


## 4. Out-of-sample prediction

For each test observation, the fitted model gives predicted probabilities:

$$
\widehat p_{t,k} = \widehat{\Pr}(Y_t=k\mid x_t).
$$

The predicted class is the class with the largest predicted probability:

$$
\hat{Y}_t = \arg\max_k \widehat p_{t,k}.
$$

In [5]:
X_test = sm.add_constant(test[["x"]], has_constant="add")
y_test = test["Y"]

p_test = res.predict(X_test)
y_hat = p_test.idxmax(axis=1).astype(int)

prediction_table = p_test
prediction_table.columns = ["p_SELL", "p_HOLD", "p_BUY"]
prediction_table["Y_hat"] = y_hat

print("Predicted probabilities and predicted classes, first 12 OOS rows:")
print(prediction_table.head(12).round(4))

accuracy = (y_hat.values == y_test.values).mean()
print("\nOOS accuracy:", round(accuracy, 4))

Predicted probabilities and predicted classes, first 12 OOS rows:
     p_SELL  p_HOLD   p_BUY  Y_hat
210  0.3720  0.2023  0.4256      2
211  0.2610  0.2086  0.5303      2
212  0.2945  0.2080  0.4975      2
213  0.3551  0.2040  0.4409      2
214  0.5677  0.1686  0.2637      0
215  0.3329  0.2058  0.4612      2
216  0.6627  0.1441  0.1933      0
217  0.5505  0.1725  0.2770      0
218  0.6041  0.1598  0.2361      0
219  0.4796  0.1868  0.3336      0
220  0.1050  0.1875  0.7075      2
221  0.8341  0.0859  0.0799      0

OOS accuracy: 0.5889


## 6. Apply the trading strategy

The strategy-return vector is filled from the predicted class. For each out-of-sample date $t$:

$$
r_t^{strategy} =\begin{cases}
r_t^f - r_t^{stock}, & \widehat Y_t = 0 \quad \text{SELL},\\
r_t^f, & \widehat Y_t = 1 \quad \text{HOLD},\\
r_t^{stock}, & \widehat Y_t = 2 \quad \text{BUY}.
\end{cases}
$$

Interpretation:

- `SELL`: short the stock and earn the risk-free return on capital, so use $r_t^f - r_t^{stock}$.
- `HOLD`: invest in the risk-free asset, so use $r_t^f$.
- `BUY`: invest in the stock, so use $r_t^{stock}$.

In [6]:
strategy = pd.Series(np.nan, index=test.index, name="r_strategy")

strategy[y_hat == 0] = test.loc[y_hat == 0, "r_f"] - test.loc[y_hat == 0, "r_stock"]
strategy[y_hat == 1] = test.loc[y_hat == 1, "r_f"]
strategy[y_hat == 2] = test.loc[y_hat == 2, "r_stock"]

strategy_table = test[["r_stock", "r_f", "Y"]].copy()
strategy_table["Y_hat"] = y_hat
strategy_table["r_strategy"] = strategy

print("Strategy table, first 20 OOS rows:")
print(strategy_table.head(20).round(5))


Strategy table, first 20 OOS rows:
     r_stock     r_f  Y  Y_hat  r_strategy
210  0.01461  0.0001  2      2     0.01461
211  0.01337  0.0001  2      2     0.01337
212  0.00647  0.0001  1      2     0.00647
213 -0.01387  0.0001  0      2    -0.01387
214 -0.01720  0.0001  0      0     0.01730
215  0.02899  0.0001  2      2     0.02899
216  0.00425  0.0001  1      0    -0.00415
217 -0.02628  0.0001  0      0     0.02638
218 -0.00234  0.0001  0      0     0.00244
219  0.02077  0.0001  2      0    -0.02067
220 -0.01296  0.0001  0      2    -0.01296
221 -0.00782  0.0001  0      0     0.00792
222 -0.00733  0.0001  0      2    -0.00733
223 -0.00409  0.0001  0      0     0.00419
224 -0.02374  0.0001  0      0     0.02384
225 -0.03105  0.0001  0      2    -0.03105
226 -0.03892  0.0001  0      0     0.03902
227 -0.01073  0.0001  0      0     0.01083
228 -0.00109  0.0001  0      0     0.00119
229 -0.01345  0.0001  0      2    -0.01345


## 7. Cumulative performance

Cumulative gross performance is computed as:

$$
V_T = \prod_{t=1}^{T}(1+r_t).
$$

We compare the strategy with buy-and-hold.

In [7]:
wealth = pd.DataFrame({
    "strategy": (1 + strategy).cumprod(),
    "buy_and_hold": (1 + test["r_stock"]).cumprod(),
    "risk_free": (1 + test["r_f"]).cumprod(),
})

print("Cumulative wealth, first 10 OOS rows:")
print(wealth.head(10).round(4))

print("\nFinal cumulative wealth:")
print(wealth.iloc[-1].round(4))

Cumulative wealth, first 10 OOS rows:
     strategy  buy_and_hold  risk_free
210    1.0146        1.0146     1.0001
211    1.0282        1.0282     1.0002
212    1.0348        1.0348     1.0003
213    1.0205        1.0205     1.0004
214    1.0381        1.0029     1.0005
215    1.0682        1.0320     1.0006
216    1.0638        1.0364     1.0007
217    1.0919        1.0091     1.0008
218    1.0945        1.0068     1.0009
219    1.0719        1.0277     1.0010

Final cumulative wealth:
strategy        1.9403
buy_and_hold    0.8300
risk_free       1.0090
Name: 299, dtype: float64


## 8. Minimal takeaways

The core implementation is:

1. Estimate multinomial logit on training data.
2. Predict out-of-sample class probabilities.
3. Convert probabilities to predicted classes using `argmax`.
4. Compare against the majority-class benchmark.
5. Fill the strategy-return vector according to the predicted class.

The trading rule is the important object:

$$
\widehat Y_t=0 \Rightarrow r_t^{strategy}=r_t^f-r_t^{stock},
\qquad
\widehat Y_t=1 \Rightarrow r_t^{strategy}=r_t^f,
\qquad
\widehat Y_t=2 \Rightarrow r_t^{strategy}=r_t^{stock}.
$$